# Programation Par Contraite - Covering Arrays avec contraintes semantiques

In [ ]:
from ortools.sat.python import cp_model
import numpy as np
import matplotlib.pyplot as plt

## Qu'est ce qu'un Covering Array ?

> Les Covering Arrays (CA) sont des matrices $N \times k$ où chaque colonne représente un paramètre (facteur) avec $v$ valeurs, telles que toute combinaison de $t$ colonnes contienne tous les $t$-uplets possibles au moins une fois.

Dit comme cela, la définition peut sembler un peu abstraite, alors voyons un exemple extrêmement simple.

Imaginons un formulaire où nous demandons la majorité du participant (Majeur ou non), son statut professionnel (Employé ou non) et sa nationalité (Française ou non). Nous avons donc 3 paramètres ($k = 3$), et chacun peut prendre exactement 2 valeurs ($v = 2$).

En théorie, il y a $2 \times 2 \times 2 = 2^3 = v^k = 8$ combinaisons possibles si on voulait tout tester. L'idée du Covering Array est de ne pas tester toutes les combinaisons globales, mais de garantir la couverture sur un groupe de $t$ colonnes. Si on prend une force $t = 2$ (*pairwise testing*), on obtient une matrice où chaque paire de colonnes contient toutes les paires de valeurs possibles au moins une fois.

$$
\text{CA} = \begin{bmatrix}
\text{Majeur} & \text{Employé} & \text{Français} \\
0 & 0 & 1 \\
1 & 0 & 0 \\
0 & 1 & 0 \\
1 & 1 & 1 
\end{bmatrix}
$$

Ici, on peut voir que si on isole deux colonnes au choix (par exemple *Majeur* et *Français*), on retrouve bien les 4 paires possibles : $(0, 0)$, $(0, 1)$, $(1, 0)$ et $(1, 1)$. 

Tous les cas globaux n'ont pas été testés (seulement 4 lignes sur 8), mais cette méthode permet de réduire grandement le nombre de tests tout en gardant une excellente capacité à détecter les bugs d'interactions.

## Contraintes Sémentiques

Vous avez peut-être relevé un problème logique dans l'exemple précédent. En effet, la troisième ligne propose un profil **Mineur (0) et Employé (1)**, ce qui est interdit par la loi (plus ou moins). 

Pour résoudre ce problème, on ajoute des **contraintes sémantiques** exprimées en logique propositionnelle : ici, $\neg(\text{Mineur} \land \text{Employé})$. 

Exclure des combinaisons rend le problème plus difficile pour le solveur. Comme certaines lignes "partagées" deviennent interdites, le solveur doit réorganiser la matrice et est parfois obligé d'**augmenter le nombre de lignes $N$** pour réussir à couvrir le reste des paires légitimes.

Avec notre contrainte, le nombre minimal de lignes passe de $N=4$ à $N=5$ :

$$
\text{CA}_{\text{contraint}} = \begin{bmatrix}
\text{Majeur} & \text{Employé} & \text{Français} \\
1 & 1 & 0 \\
1 & 0 & 1 \\
0 & 0 & 0 \\
0 & 0 & 1 \\
1 & 1 & 1
\end{bmatrix}
$$

On remarque qu'aucune ligne ne contient la combinaison interdite $(0, 1)$ sur les colonnes Majeur/Employé, mais que toutes les autres paires du système restent testées.

## Sujet

Dans ce projet, nous allons nous intéresser a la génération de Covering Array avec contraintes sémentiques et une force $t \ge 3$.